# Nano vLLM框架配套练习


相关文章：

* [推理框架极简入门：用Nano-vLLM搭建知识体系](https://zhuanlan.zhihu.com/p/2008285806222132143)

* [Nano vLLM架构介绍（英文）](https://github.com/CalvinXKY/nano-vllm/blob/main/docs/structures.md)


Author: kaiyuan

Email: kaiyuanxie@yeah.net

# 1 请求处理主要流程


## 1.1 请求序列的编码

In [14]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", use_fast=True)


In [15]:
token_ids = tokenizer.encode("hello")
print(token_ids)

[14990]


In [ ]:
token_ids = tokenizer.encode("InfraTech")
print(token_ids)

[19433, 956, 34097]


In [ ]:
token_ids = tokenizer.encode("Hi, I'm kaiyuan")
positions = list(range(len(token_ids)))
print(token_ids)
print(positions)

[13048, 11, 358, 2776, 595, 2143, 88, 10386]
[0, 1, 2, 3, 4, 5, 6, 7]


In [ ]:
token_ids = tokenizer.encode("Do you subscribe InfraTech?")
positions = list(range(len(token_ids)))
print(token_ids)
print(positions)

[5404, 498, 17963, 14921, 956, 34097, 30]
[0, 1, 2, 3, 4, 5, 6]


## 1.2 物理显存slot位置计算

In [ ]:
def get_slots(block_ids, block_size, seq_len):
  slots = []
  for block_id in block_ids[:-1]:
    start = block_id * block_size
    end = start + block_size
    slots.extend(list(range(start, end)))
  # 最后一个block
  start = block_ids[-1] * block_size
  end = start + (seq_len - (len(block_ids)-1) * block_size)
  slots.extend(list(range(start, end)))
  return slots

In [ ]:
token_ids = tokenizer.encode("Hi, I'm kaiyuan")
positions = list(range(len(token_ids)))
# 分配block 12、13给该请求， block size：4
slots = get_slots([12, 13], 4 , len(token_ids))
print(f"token_ids {token_ids}")
print(f"positions {positions}")
print(f"slots {slots}")

token_ids [13048, 11, 358, 2776, 595, 2143, 88, 10386]
positions [0, 1, 2, 3, 4, 5, 6, 7]
slots [48, 49, 50, 51, 52, 53, 54, 55]


In [ ]:
token_ids = tokenizer.encode("Do you subscribe InfraTech?")
positions = list(range(len(token_ids)))
# 分配block 20、25给该请求，block size：4
slots = get_slots([20, 25], 4 , len(token_ids))
print(f"token_ids {token_ids}")
print(f"positions {positions}")
print(f"slots {slots}")

token_ids [5404, 498, 17963, 14921, 956, 34097, 30]
positions [0, 1, 2, 3, 4, 5, 6]
slots [80, 81, 82, 83, 100, 101, 102]


In [ ]:
token_ids = tokenizer.encode("Hello, what can I do for you.")
print(f"token_ids {token_ids}")

token_ids [9707, 11, 1128, 646, 358, 653, 369, 498, 13]


## 1.3 Token ids解码

In [ ]:
ans = tokenizer.decode([9707, 11, 1128, 646, 358, 653, 369, 498, 13])
print(f"Answer: {ans}")

Answer: Hello, what can I do for you.


In [ ]:
ans = tokenizer.decode([9454, 11, 315, 3308, 13])
print(f"Answer: {ans}")

Answer: Yes, of course.


# 2 关键类与函数





## 2.1 原子类与函数

- config： 引擎配置文件
- SamplingParams： 采样参数
- Sequence：承载单个请求的相关信息

In [16]:
from dataclasses import dataclass
from copy import copy
from enum import Enum, auto
from itertools import count

@dataclass
class Config:
    model: str = "dummy"
    max_num_batched_tokens: int = 16384
    max_num_seqs: int = 512
    max_model_len: int = 4096
    gpu_memory_utilization: float = 0.9
    tensor_parallel_size: int = 1
    enforce_eager: bool = False
    eos: int = -1
    kvcache_block_size: int = 256
    num_kvcache_blocks: int = -1

    def __post_init__(self):
        assert self.kvcache_block_size % 256 == 0
        assert 1 <= self.tensor_parallel_size <= 8
        assert self.max_num_batched_tokens >= self.max_model_len

@dataclass
class SamplingParams:
    temperature: float = 1.0
    max_tokens: int = 64
    ignore_eos: bool = False

class SequenceStatus(Enum):
    WAITING = auto()
    RUNNING = auto()
    FINISHED = auto()


class Sequence:
    block_size = 4 # 方便演示，从默认256修改4
    counter = count()

    def __init__(self, token_ids: list[int], sampling_params = SamplingParams()):
        self.seq_id = next(Sequence.counter)
        self.status = SequenceStatus.WAITING
        self.token_ids = copy(token_ids)
        self.last_token = token_ids[-1]
        self.num_tokens = len(self.token_ids)
        self.num_prompt_tokens = len(token_ids)
        self.num_cached_tokens = 0
        self.block_table = []
        self.temperature = sampling_params.temperature
        self.max_tokens = sampling_params.max_tokens
        self.ignore_eos = sampling_params.ignore_eos

    def __len__(self):
        return self.num_tokens

    def __getitem__(self, key):
        return self.token_ids[key]

    @property
    def is_finished(self):
        return self.status == SequenceStatus.FINISHED

    @property
    def num_completion_tokens(self):
        return self.num_tokens - self.num_prompt_tokens

    @property
    def prompt_token_ids(self):
        return self.token_ids[:self.num_prompt_tokens]

    @property
    def completion_token_ids(self):
        return self.token_ids[self.num_prompt_tokens:]

    @property
    def num_cached_blocks(self):
        return self.num_cached_tokens // self.block_size

    @property
    def num_blocks(self):
        return (self.num_tokens + self.block_size - 1) // self.block_size

    @property
    def last_block_num_tokens(self):
        return self.num_tokens - (self.num_blocks - 1) * self.block_size

    def block(self, i):
        assert 0 <= i < self.num_blocks
        return self.token_ids[i*self.block_size: (i+1)*self.block_size]

    def append_token(self, token_id: int):
        self.token_ids.append(token_id)
        self.last_token = token_id
        self.num_tokens += 1

    def __getstate__(self):
        return (self.num_tokens, self.num_prompt_tokens, self.num_cached_tokens, self.block_table,
                self.token_ids if self.num_completion_tokens == 0 else self.last_token)

    def __setstate__(self, state):
        self.num_tokens, self.num_prompt_tokens, self.num_cached_tokens, self.block_table = state[:-1]
        if self.num_completion_tokens == 0:
            self.token_ids = state[-1]
        else:
            self.last_token = state[-1]


## 2.2 KV cache blocks数量计算

KV cache blocks数量计算的代码位于：[nanovllm/engine/model_runner.py](https://github.com/CalvinXKY/nano-vllm/blob/main/nanovllm/engine/model_runner.py)的allocate_kv_cache()函数中,通过该函数确认blocks数量上限


In [17]:
import torch
from transformers import AutoConfig

model_id = "Qwen/Qwen3-0.6B"
config = AutoConfig.from_pretrained(model_id)
# 打印出整个配置内容
print(config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

blocks数量计算演示：

In [18]:
# 数据类型，bf16 大小为2
fp_16_size = torch.empty(1, dtype=torch.bfloat16).itemsize

# 用户参数配置：
# block_size，定义block大小。 显存使用系数：gpu_memory_utilization
block_size = 4
gpu_memory_utilization = 0.95

# 从Qwen模型config里面获取数据
num_hidden_layers = config.num_hidden_layers
head_dim = config.head_dim
num_kv_heads = config.num_key_value_heads

# GPU当前状态
total = 80 * (1024 ** 3)
used = 10 * (1024 ** 3)
peak = 40 * (1024 ** 3)
current = 5 * (1024 ** 3)

# 一个block需要的数据大小
block_bytes = 2 * num_hidden_layers * block_size * num_kv_heads * head_dim * fp_16_size

# blocks总数量
num_kvcache_blocks = int(total * gpu_memory_utilization - used - peak + current) // block_bytes

# KV cache数量：
kv_cache = 2 * num_hidden_layers * num_kvcache_blocks * block_size * num_kv_heads * head_dim

print(f"num_kvcache_blocks: {num_kvcache_blocks}")
print(f"kv cache size: {kv_cache}")
print(f"kv cache mem: {kv_cache * fp_16_size / (1024 ** 3):.4} GB")

num_kvcache_blocks: 72557
kv cache size: 16642834432
kv cache mem: 31.0 GB


## 2.3 BlockManager的运行示例

### BlockManager定义

In [19]:
import xxhash
from collections import deque
import numpy as np

class Block:

    def __init__(self, block_id):
        self.block_id = block_id
        self.ref_count = 0
        self.hash = -1
        self.token_ids = []

    def update(self, hash: int, token_ids: list[int]):
        self.hash = hash
        self.token_ids = token_ids

    def reset(self):
        self.ref_count = 1
        self.hash = -1
        self.token_ids = []


class BlockManager:

    def __init__(self, num_blocks: int, block_size: int):
        self.block_size = block_size
        self.blocks: list[Block] = [Block(i) for i in range(num_blocks)]
        self.hash_to_block_id: dict[int, int] = dict()
        self.free_block_ids: deque[int] = deque(range(num_blocks))
        self.used_block_ids: set[int] = set()

    @classmethod
    def compute_hash(cls, token_ids: list[int], prefix: int = -1):
        h = xxhash.xxh64()
        if prefix != -1:
            h.update(prefix.to_bytes(8, "little"))
        h.update(np.array(token_ids).tobytes())
        return h.intdigest()

    def _allocate_block(self, block_id: int) -> Block:
        block = self.blocks[block_id]
        assert block.ref_count == 0
        block.reset()
        self.free_block_ids.remove(block_id)
        self.used_block_ids.add(block_id)
        return self.blocks[block_id]

    def _deallocate_block(self, block_id: int) -> Block:
        assert self.blocks[block_id].ref_count == 0
        self.used_block_ids.remove(block_id)
        self.free_block_ids.append(block_id)

    def can_allocate(self, seq: Sequence) -> bool:
        return len(self.free_block_ids) >= seq.num_blocks

    def allocate(self, seq: Sequence):
        assert not seq.block_table
        h = -1
        cache_miss = False
        for i in range(seq.num_blocks):
            token_ids = seq.block(i)
            h = self.compute_hash(token_ids, h) if len(token_ids) == self.block_size else -1
            block_id = self.hash_to_block_id.get(h, -1)
            if block_id == -1 or self.blocks[block_id].token_ids != token_ids:
                cache_miss = True
            if cache_miss:
                block_id = self.free_block_ids[0]
                block = self._allocate_block(block_id)
            else:
                seq.num_cached_tokens += self.block_size
                if block_id in self.used_block_ids:
                    block = self.blocks[block_id]
                    block.ref_count += 1
                else:
                    block = self._allocate_block(block_id)
            if h != -1:
                block.update(h, token_ids)
                self.hash_to_block_id[h] = block_id
            seq.block_table.append(block_id)

    def deallocate(self, seq: Sequence):
        for block_id in reversed(seq.block_table):
            block = self.blocks[block_id]
            block.ref_count -= 1
            if block.ref_count == 0:
                self._deallocate_block(block_id)
        seq.num_cached_tokens = 0
        seq.block_table.clear()

    def can_append(self, seq: Sequence) -> bool:
        return len(self.free_block_ids) >= (len(seq) % self.block_size == 1)

    def may_append(self, seq: Sequence):
        block_table = seq.block_table
        last_block = self.blocks[block_table[-1]]
        if len(seq) % self.block_size == 1:
            assert last_block.hash != -1
            block_id = self.free_block_ids[0]
            self._allocate_block(block_id)
            block_table.append(block_id)
        elif len(seq) % self.block_size == 0:
            assert last_block.hash == -1
            token_ids = seq.block(seq.num_blocks-1)
            prefix = self.blocks[block_table[-2]].hash if len(block_table) > 1 else -1
            h = self.compute_hash(token_ids, prefix)
            last_block.update(h, token_ids)
            self.hash_to_block_id[h] = last_block.block_id
        else:
            assert last_block.hash == -1

In [20]:
# 定义一个block_manager状态打印函数
def print_block_manager_info(block_manager):
    print(f"block_manager.blocks size: {len(block_manager.blocks)}")
    print(f"blocks list: {[block.block_id for block in block_manager.blocks]}")
    print(f"free_block_ids: {block_manager.free_block_ids}")
    print(f"used_block_ids: {block_manager.used_block_ids}")

### BlockManager的blocks管理逻辑演示

注意：以下代码单步仅能运行一次，重复运行得从BlockManager构建开始。

In [21]:
# 创建BlockManager
num_kvcache_blocks = 10
kvcache_block_size = 4
block_manager= BlockManager(num_kvcache_blocks, kvcache_block_size)
print_block_manager_info(block_manager)


block_manager.blocks size: 10
blocks list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
free_block_ids: deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
used_block_ids: set()


In [22]:
sampling_params = SamplingParams(temperature=0.6, max_tokens=256)
# 增加请求
token_ids = tokenizer.encode("hi, I'm kaiyuan")
seq_0 = Sequence(token_ids, sampling_params)

token_ids = tokenizer.encode("Do you subscribe InfraTech?")
seq_1 = Sequence(token_ids, sampling_params)

# 为请求申请block：
block_manager.allocate(seq_0)
block_manager.allocate(seq_1)
print_block_manager_info(block_manager)


block_manager.blocks size: 10
blocks list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
free_block_ids: deque([4, 5, 6, 7, 8, 9])
used_block_ids: {0, 1, 2, 3}


In [ ]:
# 删除请求，能够看到释放的blocks2和3到了队尾
block_manager.deallocate(seq_0)
print_block_manager_info(block_manager)

block_manager.blocks size: 10
blocks list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
free_block_ids: deque([4, 5, 6, 7, 8, 9, 1, 0])
used_block_ids: {2, 3}


In [ ]:
# 再增加一个请求，会使用队首的blocks：

token_ids = tokenizer.encode("to add new blocks.")
seq_2 = Sequence(token_ids, sampling_params)
block_manager.allocate(seq_2)
print_block_manager_info(block_manager)

block_manager.blocks size: 10
blocks list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
free_block_ids: deque([6, 7, 8, 9, 1, 0])
used_block_ids: {2, 3, 4, 5}


prefix cache匹配示例

In [ ]:
# 重复字符串输入，有相同prefix数据，复用0、1 blocks数据：
token_ids = tokenizer.encode("hi, I'm kaiyuan")
seq_3 = Sequence(token_ids, sampling_params)
block_manager.allocate(seq_3)
print_block_manager_info(block_manager)

block_manager.blocks size: 10
blocks list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
free_block_ids: deque([6, 7, 8, 9])
used_block_ids: {0, 1, 2, 3, 4, 5}


## 3 GPU运算的关键内容



## 3.1 包含KV cache的Attention计算

Attention计算使用flash_attn库的flash_attn_varlen_func和flash_attn_with_kvcache函数

在[flash-attention/flash_attn/flash_attn_interface.py](https://github.com/Dao-AILab/flash-attention/blob/main/flash_attn/flash_attn_interface.py)中找到FA计算所需的数据要求：

- Q/K/V数据格式：(total_q, nheads, headdim)，即计算decode时Q的格式为(batch_size, seqlen, nheads, headdim)
- KV cache数据格式：(num_blocks, page_block_size, nheads_k, headdim)
- cu_seqlens_q/cu_seqlens_k数据格式：(batch_size + 1,)，表示batch中各序列长度的累积值
- max_seqlen_q/k：Q/K 在 batch 中最长的序列长度
- cache_seqlens：cache 的序列长度
- block_tables数据格式：(batch_size, max_num_blocks_per_seq)，用于索引KV cache的block数据


计算步骤：

- 创建请求转换数据
- 分配逻辑存储空间
- 分配物理存储空间
- 构建FA计算数据格式
- 完成FA计算



In [ ]:
sampling_params = SamplingParams(temperature=0.6, max_tokens=256)
# 增加请求
token_ids = tokenizer.encode("hi, I'm kaiyuan")
seq_0 = Sequence(token_ids, sampling_params)

token_ids = tokenizer.encode("Do you subscribe InfraTech?")
seq_1 = Sequence(token_ids, sampling_params)
seqs = [seq_0, seq_1]

num_kvcache_blocks = 10
kvcache_block_size = 4
block_manager= BlockManager(num_kvcache_blocks, kvcache_block_size)

# 计算seq的block的最大长度，并为seq分配逻辑存储空间：
max_len = 0
for seq in seqs:
  max_len = max(len(seq.block_table), max_len)
  block_manager.allocate(seq)
print_block_manager_info(block_manager)

# 创建物理映射表：

block_tables = [seq.block_table + [-1] * (max_len - len(seq.block_table)) for seq in seqs]

print(f"物理映射表：\nblock_tables: {block_tables}")

block_manager.blocks size: 10
blocks list: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
free_block_ids: deque([4, 5, 6, 7, 8, 9])
used_block_ids: {0, 1, 2, 3}
物理映射表：
block_tables: [[0, 1], [2, 3]]


In [ ]:
# 补充模型配置参数：
num_hidden_layers = 1
num_kv_heads = 8
head_dim = 32

# 创建KV cache的物理存储空间：
kv_cache = torch.empty(2, num_hidden_layers, num_kvcache_blocks, kvcache_block_size, num_kv_heads, head_dim)
print(f"KV cache shape: {kv_cache.shape}")

# K cache与V cache数据分离：
layer_id = 0 # 假设为第0层
k_cache = kv_cache[0, layer_id]
v_cache = kv_cache[1, layer_id]

print(f"K cache shape:{k_cache.shape}")
print(f"V cache shape:{v_cache.shape}")


KV cache shape: torch.Size([2, 1, 10, 4, 8, 32])
K cache shape:torch.Size([10, 4, 8, 32])
V cache shape:torch.Size([10, 4, 8, 32])


In [ ]:
# 构建Attention计算所需要的数据：
cu_seqlens_q = [0]
cu_seqlens_k = [0]
max_seqlen_q = 0
max_seqlen_k = 0
total_seqlen = 0
for seq in seqs:
    seqlen = len(seq)
    seqlen_q = seqlen - seq.num_cached_tokens
    seqlen_k = seqlen
    total_seqlen += seqlen
    cu_seqlens_q.append(cu_seqlens_q[-1] + seqlen_q)
    cu_seqlens_k.append(cu_seqlens_k[-1] + seqlen_k)
    max_seqlen_q = max(seqlen_q, max_seqlen_q)
    max_seqlen_k = max(seqlen_k, max_seqlen_k)

print(f"total_seqlen: {total_seqlen}")
print(f"cu_seqlens_q: {cu_seqlens_q}")
print(f"cu_seqlens_k: {cu_seqlens_k}")
print(f"max_seqlen_q: {max_seqlen_q}")
print(f"max_seqlen_k: {max_seqlen_k}")


total_seqlen: 15
cu_seqlens_q: [0, 8, 15]
cu_seqlens_k: [0, 8, 15]
max_seqlen_q: 8
max_seqlen_k: 8


注： 如下计算使用了flash_atten，需要用到GPU！ 且环境支持flash_attn运行。

可以使用docker镜像，如：

```
docker pull nvcr.io/nvidia/sglang:26.01-py3
```



### prefill计算演示

In [ ]:
# 检查 CUDA 是否可用
assert torch.cuda.is_available(), "CUDA 不可用，本示例需要 GPU"

In [ ]:
from flash_attn import flash_attn_varlen_func, flash_attn_with_kvcache

# q k v数据构造
num_q_heads = num_kv_heads
q = torch.randn(total_seqlen, num_q_heads, head_dim, device='cuda', dtype=torch.float16)
k = torch.randn(total_seqlen, num_kv_heads, head_dim, device='cuda', dtype=torch.float16)
v = torch.randn(total_seqlen, num_kv_heads, head_dim, device='cuda', dtype=torch.float16)


cu_seqlens_q_cuda = torch.tensor(cu_seqlens_q, dtype=torch.int32, device='cuda')
cu_seqlens_k_cuda = torch.tensor(cu_seqlens_k, dtype=torch.int32, device='cuda')


# FA计算：
o = flash_attn_varlen_func(q, k, v,
              max_seqlen_q=max_seqlen_q, cu_seqlens_q=cu_seqlens_q_cuda,
              max_seqlen_k=max_seqlen_k, cu_seqlens_k=cu_seqlens_k_cuda,
              softmax_scale=head_dim ** -0.5, causal=True, block_table=block_tables)

### decode计算演示


In [ ]:
q = torch.randn(len(seqs), num_q_heads, head_dim, device='cuda', dtype=torch.float16)

# 创建GPU版本的 KV cache的物理存储空间：
kv_cache = torch.empty(2, num_hidden_layers, num_kvcache_blocks, kvcache_block_size, num_kv_heads, head_dim, device='cuda', dtype=torch.float16)
print(f"KV cache shape: {kv_cache.shape()}")

# K cache与V cache数据分离：
layer_id = 0 # 假设为第0层
k_cache = kv_cache[0, layer_id]
v_cache = kv_cache[1, layer_id]

# context的长度= 历史kv cache长度 + 新增seq长度
context_lens = []
for seq in seqs:
    context_lens.append(len(seq))

# FA 计算：
# q需要转为：(batch_size, seqlen, nheads, headdim)格式，seqlen = 1
o = flash_attn_with_kvcache(q.unsqueeze(1), k_cache, v_cache,
               cache_seqlens=context_lens,
               block_table=block_tables,
               softmax_scale=head_dim ** -0.5, causal=True)

##  3.2 CUDA Graph的使用

In [ ]:
# nano-vLLM中graph batchsize的支持最大值为512
max_num_seqs = 1024
max_bs = min(max_num_seqs, 512)
graph_bs = [1, 2, 4, 8] + list(range(16, max_bs + 1, 16))
print(graph_bs)

[1, 2, 4, 8, 16, 32, 48, 64, 80, 96, 112, 128, 144, 160, 176, 192, 208, 224, 240, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448, 464, 480, 496, 512]


构建一个cuda graph捕获演示示例：


In [ ]:
import torch
import torch.nn as nn
import random


# -------------------- 定义模型 --------------------
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32, 64)   # 输入特征 32 -> 64
        self.fc2 = nn.Linear(64, 32)   # 64 -> 32
        self.relu = nn.ReLU()

    def forward(self, x):
        # x 形状: [bs, 8, 32]
        x = self.fc1(x)                # [bs, 8, 64]
        x = self.relu(x)
        x = self.fc2(x)                # [bs, 8, 32]
        return x

model = SimpleModel().to(device)
model.eval()  # 推理模式，关闭 dropout/batchnorm 等随机行为

# -------------------- 为不同 batch size 捕获图 --------------------
batch_sizes = [1, 2, 4, 8, 16]
graph_pool = {}  # 字典：bs -> (graph, static_input, static_output)

# 预热 CUDA 上下文，避免捕获时包含自动调优开销
warmup_input = torch.randn(8, 8, 32, device=device)
for _ in range(3):
    _ = model(warmup_input)
torch.cuda.synchronize()

for bs in batch_sizes:
    # 创建固定的输入、输出占位张量（图会记住它们的地址）
    static_input = torch.randn(bs, 8, 32, device=device)
    static_output = torch.empty_like(static_input)  # 形状与输入相同

    # 开始捕获
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        # 在此上下文中执行的所有 CUDA 操作都会被捕获
        static_output = model(static_input)

    # 将图及相关张量保存到池中
    graph_pool[bs] = (graph, static_input, static_output)

print(f"已为 batch sizes {batch_sizes} 捕获图完成。")

# -------------------- 模拟多次推理，随机选择 batch size --------------------
num_iterations = 10
for i in range(num_iterations):
    # 随机选择一个 batch size
    bs = random.choice(batch_sizes)
    graph, static_input, static_output = graph_pool[bs]

    # 生成新的随机输入数据
    new_input = torch.randn(bs, 8, 32, device=device)

    # 将新数据复制到图使用的静态输入张量中（in-place 操作，不改变地址）
    static_input.copy_(new_input)

    # 重放图
    graph.replay()

    # 此时 static_output 已经更新为对应新输入的计算结果
    # 可以取出结果用于后续处理，例如与普通前向结果对比验证
    with torch.no_grad():
        expected_output = model(new_input)

    # 验证结果是否一致（允许微小误差）
    if torch.allclose(static_output, expected_output, atol=1e-5):
        print(f"迭代 {i+1}: bs={bs} 图重放结果与普通前向一致")
    else:
        print(f"迭代 {i+1}: bs={bs} 结果不一致！")


已为 batch sizes [1, 2, 4, 8, 16] 捕获图完成。
迭代 1: bs=4 图重放结果与普通前向一致
迭代 2: bs=2 图重放结果与普通前向一致
迭代 3: bs=4 图重放结果与普通前向一致
迭代 4: bs=1 图重放结果与普通前向一致
迭代 5: bs=4 图重放结果与普通前向一致
迭代 6: bs=1 图重放结果与普通前向一致
迭代 7: bs=2 图重放结果与普通前向一致
迭代 8: bs=4 图重放结果与普通前向一致
迭代 9: bs=4 图重放结果与普通前向一致
迭代 10: bs=8 图重放结果与普通前向一致

